# Specific Test IV – Neural Operator (FNO) Classification

**Task:** Build a classifier using a **Fourier Neural Operator (FNO)** backbone and compare against the CNN baseline from Common Test I.

**Classes:** `no` (no substructure), `sphere` (subhalo), `vort` (vortex)

**Evaluation:** ROC curves and AUC scores

---

## Strategy

### Why Fourier Neural Operators?

Standard CNNs learn **local spatial features** via small convolutional kernels. FNOs instead learn **global operators in Fourier space**, which is particularly well-suited for gravitational lensing because:

1. **Global receptive field**: Each Fourier layer sees the entire image at once via FFT, capturing the large-scale ring/arc structures that define lensing.
2. **Resolution invariance**: FNO learns in function space — the same model can theoretically operate on different resolutions.
3. **Spectral efficiency**: Lensing images have strong low-frequency structure (smooth arcs, rings). FNO's spectral truncation naturally focuses on these features.

### Architecture

- Input **lifting**: `Conv2d(1 → width)` projects the single-channel image to a higher-dimensional representation
- **4 Fourier layers**: Each applies spectral convolution (FFT → learnable weight multiply → iFFT) + a local bypass convolution + GeLU activation
- Output **projection**: Global average pool → FC head → 3-class logits
- We retain `modes=20` Fourier modes (out of 75 max for 150×150 images), keeping low-to-mid frequency components

### Key Difference from CNNs

| Aspect | CNN (ResNet-18) | FNO |
|--------|----------------|-----|
| Receptive field | Local → grows with depth | Global from layer 1 |
| Feature space | Spatial domain | Fourier (spectral) domain |
| Inductive bias | Translation equivariance | Spectral structure / smoothness |
| Parameters | ~11M | ~3-5M |
| Best for | Texture/edge patterns | Global structure / smooth fields |

---

# Part A – Common Test I Baseline (CNN)

We first reproduce the CNN baseline to enable direct comparison.

## A.1 Setup & Configuration

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix
from sklearn.preprocessing import label_binarize
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── Device Configuration ──────────────────────────────────────────────
DEVICE_OVERRIDE = None  # Set to "cuda", "mps", or "cpu" to force

if DEVICE_OVERRIDE:
    DEVICE = torch.device(DEVICE_OVERRIDE)
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

# ── Hyperparameters ───────────────────────────────────────────────────
BATCH_SIZE = 64
NUM_EPOCHS = 25
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_CLASSES = 3
NUM_WORKERS = 0
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Dataset Paths ─────────────────────────────────────────────────────
DATA_ROOT = "dataset"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR = os.path.join(DATA_ROOT, "val")
CLASS_NAMES = ["no", "sphere", "vort"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

print(f"Classes: {CLASS_NAMES}")

## A.2 Dataset & DataLoader

In [ ]:
class LensingDataset(Dataset):
    """Dataset for loading .npy strong lensing images."""

    def __init__(self, root_dir, class_names, transform=None):
        self.samples = []
        self.transform = transform
        for class_name in class_names:
            class_dir = os.path.join(root_dir, class_name)
            label = CLASS_TO_IDX[class_name]
            for fname in os.listdir(class_dir):
                if fname.endswith(".npy"):
                    self.samples.append((os.path.join(class_dir, fname), label))
        print(f"  Loaded {len(self.samples)} samples from {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = np.load(path).astype(np.float32)
        img = torch.from_numpy(img)
        if self.transform:
            img = self.transform(img)
        return img, label


# ── Transforms ─────────────────────────────────────────────────────────
train_transform_cnn = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(15),
    T.Lambda(lambda x: x.repeat(3, 1, 1)),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform_cnn = T.Compose([
    T.Lambda(lambda x: x.repeat(3, 1, 1)),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# FNO uses single-channel input
train_transform_fno = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(15),
])

val_transform_fno = None  # No transform for FNO validation

# ── CNN datasets ──────────────────────────────────────────────────────
print("Loading CNN datasets...")
train_dataset_cnn = LensingDataset(TRAIN_DIR, CLASS_NAMES, transform=train_transform_cnn)
val_dataset_cnn = LensingDataset(VAL_DIR, CLASS_NAMES, transform=val_transform_cnn)

train_loader_cnn = DataLoader(train_dataset_cnn, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
val_loader_cnn = DataLoader(val_dataset_cnn, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))

print(f"CNN → Train batches: {len(train_loader_cnn)}, Val batches: {len(val_loader_cnn)}")

## A.3 Shared Training & Evaluation Utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc="  Train", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{100.*correct/total:.1f}%")
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc="  Val  ", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / total, correct / total


def train_model(model, train_loader, val_loader, num_epochs, lr, wd, device, model_name="Model"):
    """Full training loop with early stopping. Returns history dict."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_loss = float("inf")
    best_state = None
    patience, patience_ctr = 7, 0

    print(f"\nTraining {model_name} for {num_epochs} epochs on {device}...\n")
    for epoch in range(1, num_epochs + 1):
        tl, ta = train_one_epoch(model, train_loader, criterion, optimizer, device)
        vl, va = evaluate(model, val_loader, criterion, device)
        scheduler.step()
        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["train_acc"].append(ta)
        history["val_acc"].append(va)
        lr_now = optimizer.param_groups[0]["lr"]
        marker = ""
        if vl < best_val_loss:
            best_val_loss = vl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
            marker = " ✓"
        else:
            patience_ctr += 1
        print(f"Epoch {epoch:02d}/{num_epochs} │ TrL: {tl:.4f} TrA: {100*ta:.1f}% │ "
              f"VaL: {vl:.4f} VaA: {100*va:.1f}% │ LR: {lr_now:.6f}{marker}")
        if patience_ctr >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    if best_state:
        model.load_state_dict(best_state)
        model.to(device)
        print(f"Restored best {model_name} (val_loss={best_val_loss:.4f})")
    return history


@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for images, labels in tqdm(loader, desc="Predicting", leave=False):
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


def compute_roc_auc(y_true, y_probs, class_names):
    """Compute per-class and macro ROC/AUC. Returns dict of fpr/tpr/auc."""
    n = len(class_names)
    y_bin = label_binarize(y_true, classes=list(range(n)))
    results = {}
    for i in range(n):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_probs[:, i])
        results[class_names[i]] = {"fpr": fpr, "tpr": tpr, "auc": auc(fpr, tpr)}
    # macro
    all_fpr = np.unique(np.concatenate([results[c]["fpr"] for c in class_names]))
    mean_tpr = np.zeros_like(all_fpr)
    for c in class_names:
        mean_tpr += np.interp(all_fpr, results[c]["fpr"], results[c]["tpr"])
    mean_tpr /= n
    results["macro"] = {"fpr": all_fpr, "tpr": mean_tpr, "auc": auc(all_fpr, mean_tpr)}
    return results

## A.4 Train CNN Baseline (ResNet-18)

In [ ]:
def build_resnet18(num_classes=3, pretrained=True):
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


cnn_model = build_resnet18(NUM_CLASSES).to(DEVICE)
cnn_params = sum(p.numel() for p in cnn_model.parameters())
print(f"CNN parameters: {cnn_params:,}")

cnn_history = train_model(cnn_model, train_loader_cnn, val_loader_cnn,
                          NUM_EPOCHS, LEARNING_RATE, WEIGHT_DECAY, DEVICE, "CNN (ResNet-18)")

In [ ]:
# Get CNN predictions
cnn_probs, cnn_labels = get_predictions(cnn_model, val_loader_cnn, DEVICE)
cnn_roc = compute_roc_auc(cnn_labels, cnn_probs, CLASS_NAMES)

print("\nCNN AUC Scores:")
for k, v in cnn_roc.items():
    print(f"  {k:>10s}: {v['auc']:.4f}")

---

# Part B – Fourier Neural Operator (FNO) Classifier

## B.1 FNO Architecture

This implementation follows the original FNO paper (Li et al., 2020) with adaptations for classification:
- Native complex weights on CUDA for proper gradient flow
- Hierarchical feature extraction with spatial downsampling
- Combined average + max pooling for the classification head

In [ ]:
class SpectralConv2d(nn.Module):
    """2D Fourier layer: spectral convolution via FFT.

    Follows the original FNO paper implementation:
    1. Apply 2D real FFT to input
    2. Multiply truncated Fourier modes with learnable complex weights
    3. Apply inverse FFT to return to spatial domain
    """

    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2

        # Xavier-like initialization scaled for complex weights
        scale = 1.0 / (in_channels * out_channels)
        self.weights1 = nn.Parameter(
            scale * torch.randn(in_channels, out_channels, modes1, modes2, 2))
        self.weights2 = nn.Parameter(
            scale * torch.randn(in_channels, out_channels, modes1, modes2, 2))

    def compl_mul2d(self, input_r, input_i, weights):
        """Complex multiply using real-valued storage [... , 2] for full device compat.
        (a+bi)(c+di) = (ac-bd) + (ad+bc)i
        input_r, input_i: (B, in_ch, M1, M2) real tensors
        weights: (in_ch, out_ch, M1, M2, 2) where last dim = [real, imag]
        Returns: out_r, out_i each (B, out_ch, M1, M2)
        """
        w_r = weights[..., 0]  # (in, out, M1, M2)
        w_i = weights[..., 1]
        # (B,in,M1,M2) x (in,out,M1,M2) -> (B,out,M1,M2)
        out_r = torch.einsum("bixy,ioxy->boxy", input_r, w_r) - \
                torch.einsum("bixy,ioxy->boxy", input_i, w_i)
        out_i = torch.einsum("bixy,ioxy->boxy", input_r, w_i) + \
                torch.einsum("bixy,ioxy->boxy", input_i, w_r)
        return out_r, out_i

    def forward(self, x):
        B, C, H, W = x.shape
        W_ft = W // 2 + 1

        # 2D real FFT
        x_ft = torch.fft.rfft2(x, norm='ortho')  # (B, C, H, W//2+1) complex
        x_ft_r = x_ft.real  # views, gradients flow through
        x_ft_i = x_ft.imag

        # Allocate output real and imaginary parts
        out_r = torch.zeros(B, self.out_channels, H, W_ft, device=x.device, dtype=x.dtype)
        out_i = torch.zeros(B, self.out_channels, H, W_ft, device=x.device, dtype=x.dtype)

        # Top modes (low freq height)
        r1, i1 = self.compl_mul2d(
            x_ft_r[:, :, :self.modes1, :self.modes2],
            x_ft_i[:, :, :self.modes1, :self.modes2],
            self.weights1)
        out_r[:, :, :self.modes1, :self.modes2] = r1
        out_i[:, :, :self.modes1, :self.modes2] = i1

        # Bottom modes (high freq height, wraps around)
        r2, i2 = self.compl_mul2d(
            x_ft_r[:, :, -self.modes1:, :self.modes2],
            x_ft_i[:, :, -self.modes1:, :self.modes2],
            self.weights2)
        out_r[:, :, -self.modes1:, :self.modes2] = r2
        out_i[:, :, -self.modes1:, :self.modes2] = i2

        # Reconstruct and inverse FFT
        out_ft = torch.complex(out_r, out_i)
        return torch.fft.irfft2(out_ft, s=(H, W), norm='ortho')


class FNOBlock(nn.Module):
    """FNO block: spectral conv + bypass conv + normalization + activation.
    No residual connection to force the spectral path to learn."""

    def __init__(self, in_ch, out_ch, modes1, modes2):
        super().__init__()
        self.spectral_conv = SpectralConv2d(in_ch, out_ch, modes1, modes2)
        self.bypass_conv = nn.Conv2d(in_ch, out_ch, 1)
        self.norm = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        return F.gelu(self.norm(self.spectral_conv(x) + self.bypass_conv(x)))


class FNOClassifier(nn.Module):
    """Fourier Neural Operator for image classification.

    Hierarchical architecture with spatial downsampling:
        Input (1, 150, 150)
        → Lifting: Conv2d(1, W) + BN + GELU
        → FNO Block 1: (W, 150, 150)
        → Downsample: Conv2d stride 2 → (W, 75, 75)
        → FNO Block 2: (2W, 75, 75)
        → Downsample: Conv2d stride 2 → (2W, 38, 38)
        → FNO Block 3: (2W, 38, 38)
        → FNO Block 4: (2W, 38, 38)
        → Global Avg Pool → FC head → 3 classes
    """

    def __init__(self, modes=20, width=64, num_classes=3):
        super().__init__()

        # Lifting
        self.lifting = nn.Sequential(
            nn.Conv2d(1, width, 3, padding=1),
            nn.BatchNorm2d(width),
            nn.GELU(),
        )

        # Stage 1: full resolution
        self.fno1 = FNOBlock(width, width, modes, modes)

        # Downsample 150→75
        self.down1 = nn.Sequential(
            nn.Conv2d(width, width * 2, 3, stride=2, padding=1),
            nn.BatchNorm2d(width * 2),
            nn.GELU(),
        )

        # Stage 2: half resolution (modes reduced accordingly)
        modes2 = min(modes, 37)  # max modes for 75-wide feature maps
        self.fno2 = FNOBlock(width * 2, width * 2, modes2, modes2)

        # Downsample 75→38
        self.down2 = nn.Sequential(
            nn.Conv2d(width * 2, width * 2, 3, stride=2, padding=1),
            nn.BatchNorm2d(width * 2),
            nn.GELU(),
        )

        # Stage 3 & 4: quarter resolution
        modes3 = min(modes, 19)  # max modes for 38-wide feature maps
        self.fno3 = FNOBlock(width * 2, width * 2, modes3, modes3)
        self.fno4 = FNOBlock(width * 2, width * 2, modes3, modes3)

        # Classification head
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(width * 2, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.lifting(x)
        x = self.fno1(x)
        x = self.down1(x)
        x = self.fno2(x)
        x = self.down2(x)
        x = self.fno3(x)
        x = self.fno4(x)
        return self.head(x)


# ── Instantiate ───────────────────────────────────────────────────────
FNO_MODES = 20
FNO_WIDTH = 64

fno_model = FNOClassifier(modes=FNO_MODES, width=FNO_WIDTH,
                           num_classes=NUM_CLASSES).to(DEVICE)

fno_params = sum(p.numel() for p in fno_model.parameters())
print(f"FNO parameters: {fno_params:,}")
print(f"\nArchitecture:\n{fno_model}")

# ── Sanity check: forward pass ────────────────────────────────────────
with torch.no_grad():
    dummy = torch.randn(2, 1, 150, 150, device=DEVICE)
    out = fno_model(dummy)
    print(f"\nSanity check – input: {dummy.shape} → output: {out.shape}")
    print(f"Output sample: {out[0].cpu().numpy()}")

## B.2 Train FNO

In [ ]:
# ── FNO datasets (single channel, no ImageNet normalization) ─────────
print("Loading FNO datasets...")
train_dataset_fno = LensingDataset(TRAIN_DIR, CLASS_NAMES, transform=train_transform_fno)
val_dataset_fno = LensingDataset(VAL_DIR, CLASS_NAMES, transform=val_transform_fno)

train_loader_fno = DataLoader(train_dataset_fno, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
val_loader_fno = DataLoader(val_dataset_fno, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))

# FNO trains from scratch → lower LR, more epochs
FNO_LR = 1e-3
FNO_EPOCHS = 35

fno_history = train_model(fno_model, train_loader_fno, val_loader_fno,
                          FNO_EPOCHS, FNO_LR, WEIGHT_DECAY, DEVICE, "FNO")

## B.3 FNO Evaluation

In [ ]:
fno_probs, fno_labels = get_predictions(fno_model, val_loader_fno, DEVICE)
fno_roc = compute_roc_auc(fno_labels, fno_probs, CLASS_NAMES)

print("\nFNO AUC Scores:")
for k, v in fno_roc.items():
    print(f"  {k:>10s}: {v['auc']:.4f}")

---

# Part C – Comparison: CNN vs FNO

In [ ]:
# ── Side-by-side ROC Curves ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
colors = ["#4C72B0", "#DD8452", "#55A868"]

for ax, (name, roc_data, title) in zip(axes, [
    ("CNN", cnn_roc, "CNN (ResNet-18) – ROC Curves"),
    ("FNO", fno_roc, "FNO – ROC Curves")
]):
    for i, cls in enumerate(CLASS_NAMES):
        r = roc_data[cls]
        ax.plot(r["fpr"], r["tpr"], color=colors[i], lw=2,
                label=f"{cls} (AUC={r['auc']:.4f})")
    r = roc_data["macro"]
    ax.plot(r["fpr"], r["tpr"], "k--", lw=2, label=f"macro (AUC={r['auc']:.4f})")
    ax.plot([0, 1], [0, 1], "gray", lw=1, ls=":")
    ax.set_xlabel("FPR", fontsize=12)
    ax.set_ylabel("TPR", fontsize=12)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Overlay ROC (macro only) ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
for name, roc_data, color, ls in [
    ("CNN (ResNet-18)", cnn_roc, "#4C72B0", "-"),
    ("FNO", fno_roc, "#DD8452", "--")
]:
    r = roc_data["macro"]
    ax.plot(r["fpr"], r["tpr"], color=color, ls=ls, lw=2.5,
            label=f"{name} (Macro AUC={r['auc']:.4f})")
ax.plot([0, 1], [0, 1], "gray", lw=1, ls=":")
ax.set_xlabel("FPR", fontsize=12)
ax.set_ylabel("TPR", fontsize=12)
ax.set_title("CNN vs FNO – Macro ROC Comparison", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Comparison Table ──────────────────────────────────────────────────
print("\n" + "="*65)
print(f"{'Metric':<25} {'CNN (ResNet-18)':>18} {'FNO':>18}")
print("="*65)
for cls in CLASS_NAMES + ["macro"]:
    c_auc = cnn_roc[cls]["auc"]
    f_auc = fno_roc[cls]["auc"]
    diff = f_auc - c_auc
    sign = "+" if diff > 0 else ""
    print(f"  AUC ({cls:>6s})          {c_auc:>18.4f} {f_auc:>14.4f} ({sign}{diff:.4f})")

print("-"*65)
cnn_acc = max(cnn_history["val_acc"])
fno_acc = max(fno_history["val_acc"])
print(f"  Best Val Accuracy     {100*cnn_acc:>17.1f}% {100*fno_acc:>14.1f}%")
print(f"  Parameters            {cnn_params:>18,} {fno_params:>18,}")
print(f"  Epochs Trained        {len(cnn_history['train_loss']):>18} {len(fno_history['train_loss']):>18}")
print("="*65)

## Discussion

### FNO vs CNN: Key Differences

| Aspect | CNN (ResNet-18) | FNO |
|--------|----------------|-----|
| **Feature extraction** | Local kernels, hierarchical | Global spectral convolution |
| **Receptive field** | Grows with depth (~50px at last layer) | Full image from layer 1 |
| **Inductive bias** | Translation equivariance | Spectral smoothness |
| **Parameter efficiency** | ~11M (with pretrained backbone) | ~3-5M (from scratch) |
| **Domain alignment** | General-purpose vision | Function/field operators |

### When FNO Shines
- FNOs are designed for **operator learning** — mapping between function spaces. For lensing images (which are fundamentally intensity fields governed by smooth physical processes), this is a natural fit.
- The **spectral truncation** (keeping only 20 modes out of 75) acts as an implicit regularizer, filtering out noise and focusing on the dominant spatial frequencies.
- FNOs are inherently **resolution-invariant**: the same model could process 150×150 or 300×300 images without architectural changes.

### When CNN Has the Edge
- **Pretrained weights** give CNNs a massive advantage for small datasets. Our ResNet-18 leverages millions of ImageNet images, while FNO trains from scratch.
- CNNs are better at detecting **fine local textures** that may distinguish substructure types (e.g., small perturbations in the arc).
- The PyTorch ecosystem is heavily optimized for CNNs (cuDNN, etc.).

### Implementation Notes
- Spectral convolution stores complex weights as real tensors with shape `[..., 2]` (last dim = [real, imag]). The complex multiply is done manually via `einsum`, ensuring gradients flow correctly on all devices.
- We use `norm='ortho'` in `rfft2`/`irfft2` which normalizes the transform and prevents magnitude explosion during training.
- The architecture uses **hierarchical downsampling** (150→75→38) between FNO blocks, similar to how CNNs progressively reduce spatial resolution. This creates a multi-scale spectral representation.
- No residual connections in FNO blocks — this forces the spectral path to actively learn features rather than relying on identity shortcuts.

### Interpretation
The comparison reveals whether the task is better addressed by local feature hierarchies (CNN) or global spectral analysis (FNO). In gravitational lensing, the macro structure (arcs, rings, Einstein radius) is often more discriminative than local textures, which may favor the FNO approach despite having fewer parameters.